# sandbox

> Abstractions for creating a Modal Sandbox, and interacting with it via SSH.

```mermaid
flowchart TD
    A[Create the app] --> B[Define the image]
    B --> C[Launch the sandbox]
    C --> D[Set up SSH]
```

Modal is a library that allows you to straightfowardly use cloud GPU compute programmatically in Python. No extra scripts, YAML files or whatnot needed. 

Modal allows you to run your code on a GPU in multiple ways. 

- Use a decorator like `@app.function(gpu=...)` for stateless tasks. When you run a function wrapped like this, Modal spins up a container, runs your function, and then recycles it. Good for one-off compute like inference or batch processing.
- Use `@app.cls(gpu=...)` for stateful classes. Akin to loading a model once at startup and repeatedly calling its methods. Containers stay alive between calls, and hooks like `@modal.enter()` let you set things up when the container starts.
- Or, run a persistent interactive shell via `modal.Sandbox`. This gives you a full Linux environment that runs continuously. You can SSH in, install packages, and run code interactively.

The third option is what `solveit-modal` will be using.

In [ ]:
#| default_exp sandbox

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| hide
enable_mermaid()

<script type="module">
if (window.mermaid) mermaid.run()
else {
    import('https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs').then(m => {
        window.mermaid = m.default;
        window.mermaid.run();
        htmx.onLoad(elt => {
            if (elt.matches('div.mermaid, pre.mermaid') || htmx.findAll(elt, 'div.mermaid, pre.mermaid')) window.mermaid.run();
        });
    });
}</script>

In [ ]:
#| export
import modal

In [ ]:
#| export
def mk_app(
       name:str  # App name to look up or create 
) -> modal.App:
    "Look up a Modal App by name, creating it if missing."
    return modal.App.lookup(name, create_if_missing=True)

An app is like a folder for our projects, or sandboxes in this case. When we create a sandbox, we'll register it under this app. An app can contain multiple sandboxes.

In [ ]:
app = mk_app('solveit-sandbox'); app

<modal.app.App>

In [ ]:
#| export
def mk_img(
    pips:list, # pip packages to have installed
    apts:list  # apt packages to have installed
) -> modal.Image:
    "Create a Modal Image with system + Python packages."
    return modal.Image.debian_slim().apt_install(apts).pip_install(pips)

In [ ]:
modal.Image.debian_slim?

````python
def blocking_debian_slim(
    python_version:Optional=None, force_build:bool=False
)->_Image:

````



````
Default image, based on the official `python` Docker images.
````



**File:** `~/.local/lib/python3.12/site-packages/modal/image.py`

**Type:** function

The image describes our Linux environment setup. By default, a build of Debian is used, and we can choose which apt packages, pip packages, or other packages we can have pre-installed.

In [ ]:
img = mk_img(pips=['amogus'], apts=['openssh-server']); img

Image(<function _Image.pip_install.<locals>.build_dockerfile>)

In [ ]:
#| export
def mk_vol(
    name:str # Volume name to look up or create
) -> dict:
    "Look up a Modal Volume by name, creating it if missing."
    return {'/data': modal.Volume.from_name(name, create_if_missing=True)}

In [ ]:
vol = mk_vol('solveit-volume'); vol

{'/data': modal.Volume.from_name('solveit-volume')}

We can also create a volume in which we can persistently store data every time the sandbox is loaded.

In [ ]:
#| export
import logging

In [ ]:
#| export
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s | %(asctime)s')
log = logging.getLogger(__name__)

In [ ]:
log.info('haha')

INFO - haha | 2026-06-17 06:18:55,225


In [ ]:
#| export
def mk_sandbox(
    app    :modal.App,    # Modal App to register the Sandbox with
    img    :modal.Image,  # Image for the Sandbox environment
    vol    :dict,         # Volume for the Sandbox environment
    timeout:int,          # auto-terminate after this many seconds
    gpu    :str,          # GPU type (e.g. "T4", "A100") or None
    secrets:dict,         # Sandbox secrets
) -> modal.Sandbox:
    "Create a Modal Sandbox with optional GPU, SSH, and Volumes."
    log.info('∞ creating sandbox; this may take 5-10 minutes if you are creating this sandbox for the first time...')
    sb = modal.Sandbox.create('sleep', 'infinity',  # <1>
        app=app, image=img, volumes=vol,
        unencrypted_ports=[22], 
        timeout=timeout, gpu=gpu,  # <2>
        secrets=[modal.Secret.from_dict(secrets)])
    log.info('✔ sandbox ready')
    return sb

1. `sleep infinity` keeps the sandbox alive until it times out or is stopped.
2. Port 22 is exposed as a raw TCP tunnel so we can SSH into it.

It's now time to construct our sandbox with our defined app, image, and volume on Modal.

In [ ]:
sb = mk_sandbox(app, img, vol, timeout=600, gpu='T4', secrets={}); sb

INFO - ∞ creating sandbox; this may take 5-10 minutes if you are creating this sandbox for the first time... | 2026-06-17 06:19:52,242


INFO - ✔ sandbox ready | 2026-06-17 06:19:58,555


Sandbox()

Our sandbox is now running. To connect, we need to fetch the tunnel opened on port 22.

In [ ]:
#| export
def get_tunnel(
    sb: modal.Sandbox  # Sandbox with an exposed TCP tunnel
) -> tuple[str,int]:
    "Get unencrypted host and port for a Sandbox's TCP tunnel."
    t = sb.tunnels()[22]
    host, port = t.unencrypted_host, t.unencrypted_port
    log.info(f'✔ gotten tunnel: {host}:{port}')
    return host, port

In [ ]:
host,port = get_tunnel(sb); host,port

INFO - ✔ gotten tunnel: r438.modal.host:42091 | 2026-06-17 06:20:22,784


('r438.modal.host', 42091)

In [ ]:
#| export
import os

In [ ]:
#| export
def get_pubkey() -> str:
    "SSH public key from environment."
    if 'SSH_PUBKEY' not in os.environ:
        raise KeyError("SSH_PUBKEY not set — cannot connect to sandbox via SSH")
    return os.environ['SSH_PUBKEY']

We fetch SolveIt's public key from our environment and inject it into the sandbox's authorized keys to grant SSH access.

In [ ]:
get_pubkey()

'ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAID5fw9cXzFDJDm8mCuWUOesThxZZfdlPEsKmMSYrDmmH solveit@8321abd9ea3b'

In [ ]:
#| export
def inject_pubkey(
    sb    :modal.Sandbox,  # Sandbox to inject the key into
    pubkey:str             # SSH public key string
):
    "Inject an SSH public key into the Sandbox's authorized_keys."
    sb.exec('mkdir', '-p', '/root/.ssh')
    sb.exec('bash', '-c', f'echo {pubkey} > /root/.ssh/authorized_keys')
    log.info('✔ public key injected')

In [ ]:
inject_pubkey(sb, get_pubkey())

INFO - ✔ public key injected | 2026-06-17 06:21:00,117


We can now start our SSH service.

In [ ]:
#| export
from time import sleep

In [ ]:
#| export
def start_ssh(
    sb: modal.Sandbox  # Sandbox with SSH server installed
):
    "Start SSH service in the Sandbox, waiting up to ~6 seconds for it to come online."
    sb.exec('service', 'ssh', 'start')
    for _ in range(20):
        proc = sb.exec('service', 'ssh', 'status')
        if 'fail' in proc.stdout.read(): sleep(0.3)
        elif 'run' in proc.stdout.read():
            log.info('✔ started ssh service')
            return
    log.error('✖ failed to start ssh service')

In [ ]:
start_ssh(sb)

INFO - ✔ started ssh service | 2026-06-17 06:21:29,500


In [ ]:
#| export
from fastcore.all import *

In [ ]:
?run

````python
def run(
    cmd, rest:VAR_POSITIONAL, same_in_win:bool=False, ignore_ex:bool=False, as_bytes:bool=False, stderr:bool=True
):

````



````
Pass `cmd` (splitting with `shlex` if string) to `subprocess.run`; return `stdout`; raise `IOError` if fails
````



**File:** `/usr/local/lib/python3.12/site-packages/fastcore/xtras.py`

**Type:** function

Crafting a function to allow us to easily run bash commands in the sandbox.

In [ ]:
#| export
from typing import Callable

In [ ]:
#| export
def mk_ssh(
    host: str,  # Tunnel host
    port: int,  # Tunnel port
) -> Callable:
    "Return an SSH function for the given Modal tunnel."
    return lambda *cmd: run('ssh', '-p', str(port),
        '-o', 'StrictHostKeyChecking=no',
        '-o', 'UserKnownHostsFile=~/.ssh/known_hosts',
        '-o', 'ConnectTimeout=10',
        f'root@{host}', *cmd)

In [ ]:
ssh = mk_ssh(host,port); ssh

<function __main__.mk_ssh.<locals>.<lambda>(*cmd)>

In [ ]:
#| export
def verify_ssh(
    ssh: Callable  # SSH function returned by mk_ssh
):
    "Verify SSH connection to Sandbox and print system info."
    h, u, w = ssh('hostname; uname -srmo; whoami').splitlines()[:3]
    sys_name, ver, arch, os_name = u.split()
    gpu = ssh('nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo "no GPU"').strip()
    print(f'System: {sys_name}')
    print(f'Hostname: {h}')
    print(f'User: {w}')
    print(f'Kernel: {ver}')
    print(f'Architecture: {arch}')
    print(f'OS Type: {os_name}')
    print(f'GPU: {gpu}')

In [ ]:
verify_ssh(ssh)

System: Linux
Hostname: modal
User: root
Kernel: 4.4.0
Architecture: x86_64
OS Type: GNU/Linux
GPU: Tesla T4


And SolveIt has a connection to Modal! But we're not quite done yet. What's left is spinning up an IPython kernel on Modal, and then swapping out SolveIt's IPython kernel with the one started on Modal. See the [ipyfernel module](#02_ipyfernel.ipynb) for that.

In [ ]:
sb.terminate()

## Additional helper functions

Additional functions to help you customize your Modal sandbox instance.

In [ ]:
#| export
def get_apts() -> list[str]:
    "List installed system package names."
    return run('dpkg-query -W -f=\'${Package}\\n\'').splitlines()

In [ ]:
get_apts()[:10]

['adduser',
 'alsa-topology-conf',
 'alsa-ucm-conf',
 'apt',
 'aria2',
 'autoconf',
 'automake',
 'autotools-dev',
 'base-files',
 'base-passwd']

In [ ]:
#| export
def get_pips() -> list[str]:
    "List installed Python packages (name==version)."
    return [l for l in run('pip freeze').splitlines() if not l.startswith(('-e', '@'))]

In [ ]:
get_pips()[:10]

['accelerate==1.13.0',
 'anki==25.9.2',
 'beartype==0.22.5',
 'blis==1.3.0',
 'catalogue==2.0.10',
 'cbor2==5.8.0',
 'cloudpathlib==0.23.0',
 'cloudpickle==3.1.2',
 'confection==0.1.5',
 'coolname==2.2.0']

In [ ]:
#| export
import socket, os

In [ ]:
#| export
def in_modal() -> str: 
    'Checks whether current environment is Modal or not.'
    return socket.gethostname() == 'modal'

In [ ]:
in_modal()

False

In [ ]:
#| export
def in_solveit(): 
    'Checks whether current environment is SolveIt or not.'
    return bool(os.environ.get('IN_SOLVEIT'))

In [ ]:
in_solveit()

True

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()